# SKINO study — Kaggle GPU runner (distributed)

Runs the parameter-matched neural-operator comparison on Kaggle's **2× T4** GPUs.

## Before you run

1. **Settings → Accelerator → `GPU T4 x2`**  (and **Internet → On**)
2. **Add Data → Upload** a dataset containing the folders `track2/`, `skino/`, `validation/`
   from the repo. Note the dataset slug — it mounts at `/kaggle/input/<slug>/`.
3. Set `DATASET` below to that slug.
4. *(optional, for Drive upload)* **Add-ons → Secrets** → add
   `GDRIVE_SA_JSON` (service-account JSON) and `GDRIVE_FOLDER_ID`.
   Without these the notebook still produces a downloadable zip.

## Scaling across sessions

The job matrix is embarrassingly parallel. Open **N sessions** and give each a
different `SHARD` (`0..N-1`) with the same `NUM_SHARDS`. Nothing is duplicated
or lost, and re-running a shard resumes it (finished jobs are skipped).

In [ ]:
# ---- configure ----
DATASET     = "skino-code"   # <-- your Kaggle dataset slug
STAGE       = "1d"           # "1d" -> "2d" -> "3d" (run stages in order)
SEEDS       = [0, 1, 2]
SHARD       = 0              # this session's index
NUM_SHARDS  = 1              # total sessions you are running
CONFIGS     = None           # None = 6 core configs; "full" = all 17
SAVE_FIELDS = True           # needed for the actual-vs-predicted figures

import torch, os
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", N_GPU)
for i in range(N_GPU):
    print("  ", i, torch.cuda.get_device_name(i))
assert N_GPU > 0, "No GPU. Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# ---- stage the code into a writable dir (/kaggle/input is read-only) ----
import shutil, os, sys
SRC = f"/kaggle/input/{DATASET}"
DST = "/kaggle/working/skino_run"
os.makedirs(DST, exist_ok=True)
for pkg in ("track2", "skino", "validation"):
    s = os.path.join(SRC, pkg)
    if not os.path.isdir(s):
        raise SystemExit(f"missing {pkg} in {SRC} — check the dataset contents")
    shutil.copytree(s, os.path.join(DST, pkg), dirs_exist_ok=True)
os.chdir(DST)
sys.path.insert(0, DST)
print("staged:", os.listdir(DST))

In [ ]:
# ---- sanity: solvers correct + every family builds at matched budget ----
!python -m track2.pde_solvers

from track2.models import build_matched
for fam in ["skino", "skino_strict", "fno", "tfno", "ufno"]:
    _, n, wr = build_matched(fam, 1, 1, 64, 0.005, 25000)
    print(f"  {fam:13s} {n:8,d} params  width={wr}")

In [ ]:
# ---- what this session will run ----
cfg_arg = [] if CONFIGS is None else (["--configs", "full"] if CONFIGS == "full"
                                      else ["--configs", *CONFIGS])
seed_arg = ["--seeds", *map(str, SEEDS)]
base = ["-m", "track2.launcher", "--stage", STAGE, *seed_arg, *cfg_arg,
        "--shard", str(SHARD), "--num-shards", str(NUM_SHARDS)]

import subprocess, sys
print(subprocess.run([sys.executable, *base, "--plan"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ---- RUN: one concurrent job per GPU ----
# --gpus pins each worker to its own device via CUDA_VISIBLE_DEVICES,
# so 2x T4 really does 2 jobs at once.
import subprocess, sys, time
cmd = [sys.executable, *base, "--device", "cuda",
       "--jobs", str(N_GPU), "--gpus", str(N_GPU)]
if SAVE_FIELDS:
    cmd.append("--save-fields")
print(" ".join(cmd), "\n")

t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print(f"\nexit={p.returncode}  elapsed={(time.time()-t0)/60:.1f} min")

In [ ]:
# ---- merge this session's per-job files ----
# (If you ran several sessions, do the merge after collecting every shard's
#  output, otherwise the merged file only covers this shard.)
!python -m track2.launcher --merge

In [ ]:
# ---- quick look at what this shard produced ----
import glob, json
for f in sorted(glob.glob("track2/results_paper/paper_*_sub_*.json")):
    d = json.load(open(f)); m = d["_meta"]
    for k, v in d.items():
        if k == "_meta":
            continue
        cp = str(m["checkpoints"][-1])
        r = v["metrics"].get(cp, {})
        print(f"{m['problem']:10s} {k:16s} s{m['seed']} "
              f"UH={v['usable_horizon']:4d} "
              f"rms={r.get('rel_rms', float('nan')):.4g} [{r.get('verdict','?')}]")

In [ ]:
# ---- ship results: Google Drive if secrets are set, else a downloadable zip ----
!pip -q install google-api-python-client google-auth 2>/dev/null
!python -m track2.sync_drive --results-dir track2/results_paper --name skino_{STAGE}_shard{SHARD}

# Anything left in /kaggle/working is downloadable from the Output pane.
import shutil, os
shutil.make_archive(f"/kaggle/working/skino_{STAGE}_shard{SHARD}", "zip",
                    "track2/results_paper")
print("zip ready:", [f for f in os.listdir("/kaggle/working") if f.endswith(".zip")])

## Next stages

Only advance when the previous stage looks good (best model reaching
`rel RMS < 0.1` with verdict `good`, and seed spread small compared with the
gaps between models):

| stage | set `STAGE` to | notes |
|---|---|---|
| 1-D ladder | `"1d"` | 5 equations, 512 trajectories, 500-step rollout |
| 2-D wave | `"2d"` | 96 trajectories, 250-step rollout |
| 3-D wave | `"3d"` | 48 trajectories, 150-step rollout — GPU matters most here |

Then, locally, after collecting every shard:

```bash
python -m track2.launcher --merge
python -m track2.seed_analysis     # mean +/- std + significance
python -m track2.paper_analysis    # tables + figures
```